### Data Scraper for extracting from USA Data Centers

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import urljoin
import time

BASE_URL = "https://www.datacentermap.com"
START_URL = "https://www.datacentermap.com/usa/"

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0"
})

data = []

def get_soup(url):
    response = session.get(url)
    response.raise_for_status()
    return BeautifulSoup(response.text, "html.parser")

# 1️⃣ Get all state links
soup = get_soup(START_URL)

state_links = [
    urljoin(BASE_URL, a["href"])
    for a in soup.select("a")
    if a.get("href", "").startswith("/usa/") 
]

print(f"Found {len(state_links)} states")

for state_url in state_links:
    state_name = state_url.split("/")[-2].capitalize()
    print("State:", state_name)

    state_soup = get_soup(state_url)

    # 2️⃣ Get market/city links
    city_links = [
        urljoin(BASE_URL, a["href"])
        for a in state_soup.select("a")
        if a.get("href", "").startswith(state_url.replace(BASE_URL, ""))
        and a["href"].count("/") == 4
    ]

    print(f"Found {len(city_links)} cities")

    for city_url in city_links:
        city_soup = get_soup(city_url)

        # find all the “card” blocks
        cards = city_soup.select("a.ui.card.card1")

        for card in cards:
            # Datacenter name
            name_tag = card.select_one(".header")
            name = name_tag.get_text(strip=True) if name_tag else ""

            # Description block
            description_block = card.select_one(".description")
            description_text = description_block.get_text(separator=" | ", strip=True) if description_block else ""

            # Extra content block
            extra_block = card.select_one(".extra.content.darkgrey")
            extra_text = extra_block.get_text(strip=True) if extra_block else ""

            # Combine into one string
            metadata = f"{description_text} | {extra_text}" if description_text or extra_text else ""

            data.append({
                "state": state_name,
                "datacenter_name": name,
                "metadata": metadata
            })
    
            print(f"{state_name} {name} {metadata}")
    time.sleep(1)  # be polite
    
# Save to CSV
datacenters = pd.DataFrame(data)
datacenters.to_csv("../../data/us_datacenters.csv", index=False)

print("Done.")

### Reverse Search Opening Dates

In [ ]:
!pip install pyperclip

In [ ]:
import pandas as pd
import json

datacenters = pd.read_csv("../../data/us_datacenters.csv")
print(datacenters.shape)
print(datacenters.columns)

# clean data first 
states = pd.read_json("../../data/state_map.json", orient="index", typ='series')
valid_states = set(states.values)
datacenters = datacenters[datacenters['state'].notna() & datacenters['state'].str.strip().isin(valid_states)]

datacenters['datacenter_name'] = datacenters['datacenter_name'].str.strip().str.replace(r'\s+', ' ', regex=True).str.title()
df = datacenters[datacenters['datacenter_name'] != ""].drop_duplicates(subset=['datacenter_name'])
print(datacenters.shape)

# prep master dataframe
datacenters_dated = pd.DataFrame(columns=list(datacenters.columns) + ["opening_year"])

(4638, 3)
Index(['state', 'datacenter_name', 'metadata'], dtype='object')
(4158, 3)


In [ ]:
text = "I have a CSV listing datacenters with columns for name, state, operator, \
            and address. For each datacenter, determine its operational opening year as a \
            datacenter (not building construction). If the opening year is not explicitly \
            public, infer it using publicly available information, and cite the source URL \
            for that inference. Output a CSV with all original columns, plus two new columns:\
            'Year Opened' (YYYY) and 'Source URL'. For sites with multiple phases, use the \
            earliest operational year unless a phase is specifically documented. Include \
            both explicit and inferred years with sources."

In [ ]:
import os 
import pyperclip 

# split into batches
batch_size = 50
for start in range(0, len(datacenters), batch_size):
    batch = datacenters.iloc[start:start+batch_size]

    # build prompt for ChatGPT
    prompt = []
    
    for index, row in batch.iterrows():
        prompt.append(
            f"{row['datacenter_name']},{row['state']},{row['metadata']}"
        )
        
    prompt_text = "\n".join(prompt)
    
    # clear screen
    os.system('cls' if os.name == 'nt' else 'clear')

    # print batch with prompt 
    pyperclip.copy(prompt_text)
    print(prompt_text)
    input("\n--- Batch copied to clipboard, get response, press enter to continue")

print("All batches ready.")

### Clean output from date extractor

In [ ]:
import csv

input_file = "../../data/datacenters.csv"
output_file = "../../data/datacenters_cleaned.csv"

correct_columns = 6

def fix_line(line):
    """Fix a line by combining fields until quotes are balanced."""
    parts = []
    current = ''
    in_quotes = False
    
    for char in line:
        if char == '"':
            in_quotes = not in_quotes
            current += char
        elif char == ',' and not in_quotes:
            parts.append(current.strip())
            current = ''
        else:
            current += char
    parts.append(current.strip())
    return parts

with open(input_file, 'r', encoding='utf-8') as infile, \
     open(output_file, 'w', newline='', encoding='utf-8') as outfile:
    
    writer = csv.writer(outfile)
    
    for line in infile:
        line = line.strip()
        if not line:
            continue
        
        row = fix_line(line)
        
        if len(row) > correct_columns:
            # Merge extra columns into the first column
            row = [','.join(row[:len(row) - (correct_columns - 1)])] + row[-(correct_columns - 1):]
        elif len(row) < correct_columns:
            # Pad missing columns
            row += [''] * (correct_columns - len(row))
        
        writer.writerow(row)

### Verify CSV can be read into pandas 

In [10]:
datacenters_raw = pd.read_csv('../../data/datacenters_cleaned.csv')
datacenters_raw[:5]

,Name,State,Operator,Address,Year Opened,Source URL
0,2323 Bryan Street (Dfw10),Texas,Digital Realty,"""2323 Bryan St, Dallas, TX 75201, USA""",1989,https://baxtel.com/data-center/2323-bryan-stre...
1,4025 Midway Rd (Dfw11),Texas,Digital Realty,"""4025 Midway Rd, Carrollton, TX 75007, USA""",2000,https://baxtel.com/data-center/digital-realty-...
2,H5Colo Inc.,Texas,H5 Colo,"""12712 Park Central Dr, Dallas, TX 75251, USA""",2010,https://www.h5colo.com/about
3,1232 Alma Road (Dfw16),Texas,Digital Realty,"""1232 Alma Rd, Richardson, TX 75081, USA""",2001,https://baxtel.com/data-center/digital-realty-...
4,8435 Stemmons Freeway (Dfw36),Texas,Digital Realty,"""8435 Stemmons Fwy, Dallas, TX 75247, USA""",2001,https://baxtel.com/data-center/8435-stemmons-f...
